In [2]:
# Welcome to your new notebook
# Type here in the cell editor to add code!

from pyspark.sql import functions as F
from pyspark.sql import types as T

# Load Bronze data
df_bronze = spark.table("bronze_ed_quarterly")

print("Bronze records:", df_bronze.count())
print("Bronze columns:", len(df_bronze.columns))

display(df_bronze.limit(5))


StatementMeta(, 4abdc89a-5642-4909-949b-16224ba212a9, 4, Finished, Available, Finished, False)

Bronze records: 6930
Bronze columns: 17


SynapseWidget(Synapse.DataFrame, 8ff30b80-9659-4382-9f12-c89b756a1da7)

In [3]:

# Create Silver DataFrame
df_silver = df_bronze

# Integer columns
integer_columns = [
    "facility_code",
    "number_of_attendances",
    "left_after_treatment_count",
    "did_not_wait_count"
]

# Decimal/numeric clinical indicators
numeric_columns = [
    "attendance_variation_pct",
    "seen_within_target_pct",
    "median_wait_minutes",
    "did_not_wait_pct",
    "admitted_from_ed_pct",
    "admissions_within_4h_pct",
    "ed_stay_within_4h_pct",
    "left_after_treatment_pct"
]

# Convert integers using try_cast
for col_name in integer_columns:
    df_silver = df_silver.withColumn(
        col_name,
        F.expr(f"try_cast(`{col_name}` AS INT)")
    )

# Convert percentages and wait times
for col_name in numeric_columns:
    df_silver = df_silver.withColumn(
        col_name,
        F.expr(f"try_cast(trim(`{col_name}`) AS DOUBLE)")
    )

# Standardise quarter dates: Dec-23 -> 2023-12-01
df_silver = df_silver.withColumn(
    "quarter_end_month",
    F.to_date(
        F.initcap(F.trim(F.col("reporting_quarter"))),
        "MMM-yy"
    )
)

# Convert triage category to a clean string
df_silver = df_silver.withColumn(
    "triage_category",
    F.upper(F.trim(F.col("triage_category")))
)

print("Silver transformation prepared.")

df_silver.printSchema()
display(df_silver.limit(5))


StatementMeta(, 4abdc89a-5642-4909-949b-16224ba212a9, 5, Finished, Available, Finished, False)

Silver transformation prepared.
root
 |-- facility_code: integer (nullable = true)
 |-- facility_name: string (nullable = true)
 |-- reporting_quarter: string (nullable = true)
 |-- triage_category: string (nullable = true)
 |-- number_of_attendances: integer (nullable = true)
 |-- attendance_variation_pct: double (nullable = true)
 |-- seen_within_target_pct: double (nullable = true)
 |-- median_wait_minutes: double (nullable = true)
 |-- did_not_wait_pct: double (nullable = true)
 |-- admitted_from_ed_pct: double (nullable = true)
 |-- admissions_within_4h_pct: double (nullable = true)
 |-- ed_stay_within_4h_pct: double (nullable = true)
 |-- left_after_treatment_count: integer (nullable = true)
 |-- left_after_treatment_pct: double (nullable = true)
 |-- did_not_wait_count: integer (nullable = true)
 |-- median_wait_traffic_light: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- quarter_end_month: date (nullable = true)



SynapseWidget(Synapse.DataFrame, 48521584-4c54-42fa-9c43-394c6ed7d154)

In [4]:

# Check for values that became NULL during numeric conversion
# even though the original Bronze field was not NULL

columns_to_check = integer_columns + numeric_columns

conversion_results = []

for col_name in columns_to_check:

    invalid_count = (
        df_bronze.alias("b")
        .join(
            df_silver.alias("s"),
            on=["facility_code", "reporting_quarter",
                "triage_category", "source_file"],
            how="inner"
        )
    ) if False else None

    # Compare Bronze and Silver by using original and cast expressions
    invalid_rows = df_bronze.filter(
        F.col(col_name).isNotNull()
        &
        F.expr(
            f"try_cast(trim(`{col_name}`) AS DOUBLE) IS NULL"
        )
    ).count()

    conversion_results.append(
        (col_name, invalid_rows)
    )

conversion_check = spark.createDataFrame(
    conversion_results,
    ["column_name", "non_numeric_values"]
)

display(conversion_check.orderBy(
    F.desc("non_numeric_values")
))


StatementMeta(, 4abdc89a-5642-4909-949b-16224ba212a9, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2fc2c436-33c5-45bd-831d-80244b74b0b2)

In [5]:

from pyspark.sql import functions as F

# Columns that failed numeric conversion
columns_to_check = [
    "did_not_wait_pct",
    "left_after_treatment_pct",
    "admissions_within_4h_pct",
    "admitted_from_ed_pct",
    "attendance_variation_pct",
    "ed_stay_within_4h_pct",
    "seen_within_target_pct",
    "median_wait_minutes"
]

# Inspect original values that cannot be converted
for col_name in columns_to_check:

    print(f"\nCOLUMN: {col_name}")
    print("-" * 50)

    invalid_values = (
        df_bronze
        .filter(
            F.col(col_name).isNotNull()
            &
            F.expr(
                f"try_cast(trim(`{col_name}`) AS DOUBLE) IS NULL"
            )
        )
        .groupBy(col_name)
        .agg(F.count("*").alias("frequency"))
        .orderBy(F.desc("frequency"))
    )

    invalid_values.show(20, truncate=False)


StatementMeta(, 4abdc89a-5642-4909-949b-16224ba212a9, 7, Finished, Available, Finished, False)


COLUMN: did_not_wait_pct
--------------------------------------------------
+----------------+---------+
|did_not_wait_pct|frequency|
+----------------+---------+
|-               |857      |
+----------------+---------+


COLUMN: left_after_treatment_pct
--------------------------------------------------
+------------------------+---------+
|left_after_treatment_pct|frequency|
+------------------------+---------+
|-                       |731      |
+------------------------+---------+


COLUMN: admissions_within_4h_pct
--------------------------------------------------
+------------------------+---------+
|admissions_within_4h_pct|frequency|
+------------------------+---------+
|-                       |341      |
+------------------------+---------+


COLUMN: admitted_from_ed_pct
--------------------------------------------------
+--------------------+---------+
|admitted_from_ed_pct|frequency|
+--------------------+---------+
|-                   |217      |
+--------------------+

In [6]:

from pyspark.sql import functions as F

# Start again from Bronze to ensure reproducibility
df_silver = spark.table("bronze_ed_quarterly")

integer_columns = [
    "facility_code",
    "number_of_attendances",
    "left_after_treatment_count",
    "did_not_wait_count"
]

numeric_columns = [
    "attendance_variation_pct",
    "seen_within_target_pct",
    "median_wait_minutes",
    "did_not_wait_pct",
    "admitted_from_ed_pct",
    "admissions_within_4h_pct",
    "ed_stay_within_4h_pct",
    "left_after_treatment_pct"
]

# Convert integer columns
for c in integer_columns:
    df_silver = df_silver.withColumn(
        c,
        F.expr(f"try_cast(trim(`{c}`) AS INT)")
    )

# Convert '-' to NULL, then cast to numeric
for c in numeric_columns:
    df_silver = df_silver.withColumn(
        c,
        F.when(
            F.trim(F.col(c)) == "-",
            F.lit(None).cast("double")
        ).otherwise(
            F.expr(f"try_cast(trim(`{c}`) AS DOUBLE)")
        )
    )

# Standardise triage values
df_silver = df_silver.withColumn(
    "triage_category",
    F.upper(F.trim(F.col("triage_category")))
)

# Standardise reporting quarter
df_silver = df_silver.withColumn(
    "quarter_end_month",
    F.to_date(
        F.initcap(F.trim(F.col("reporting_quarter"))),
        "MMM-yy"
    )
)

print("Silver transformation completed.")
print("Total records:", df_silver.count())
print("Total columns:", len(df_silver.columns))

df_silver.printSchema()


StatementMeta(, 4abdc89a-5642-4909-949b-16224ba212a9, 8, Finished, Available, Finished, False)

Silver transformation completed.
Total records: 6930
Total columns: 18
root
 |-- facility_code: integer (nullable = true)
 |-- facility_name: string (nullable = true)
 |-- reporting_quarter: string (nullable = true)
 |-- triage_category: string (nullable = true)
 |-- number_of_attendances: integer (nullable = true)
 |-- attendance_variation_pct: double (nullable = true)
 |-- seen_within_target_pct: double (nullable = true)
 |-- median_wait_minutes: double (nullable = true)
 |-- did_not_wait_pct: double (nullable = true)
 |-- admitted_from_ed_pct: double (nullable = true)
 |-- admissions_within_4h_pct: double (nullable = true)
 |-- ed_stay_within_4h_pct: double (nullable = true)
 |-- left_after_treatment_count: integer (nullable = true)
 |-- left_after_treatment_pct: double (nullable = true)
 |-- did_not_wait_count: integer (nullable = true)
 |-- median_wait_traffic_light: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- quarter_end_month: date (nullable = true)


In [7]:

# Check percentage values outside the valid range

percentage_columns = [
    "seen_within_target_pct",
    "did_not_wait_pct",
    "admitted_from_ed_pct",
    "admissions_within_4h_pct",
    "ed_stay_within_4h_pct",
    "left_after_treatment_pct"
]

results = []

for c in percentage_columns:
    stats = df_silver.agg(
        F.min(c).alias("minimum"),
        F.max(c).alias("maximum"),
        F.sum(
            F.when(
                F.col(c).isNotNull() &
                ((F.col(c) < 0) | (F.col(c) > 100)),
                1
            ).otherwise(0)
        ).alias("invalid_count")
    ).first()

    results.append((
        c,
        stats["minimum"],
        stats["maximum"],
        stats["invalid_count"]
    ))

validation_df = spark.createDataFrame(
    results,
    ["column", "minimum", "maximum", "invalid_count"]
)

display(validation_df)


StatementMeta(, 4abdc89a-5642-4909-949b-16224ba212a9, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2d2a6ee6-cade-4f52-967b-a2a489ac4391)

In [8]:

from pyspark.sql import functions as F

# ---------------------------------------
# Check 1: Reporting dates
# ---------------------------------------

invalid_dates = df_silver.filter(
    F.col("quarter_end_month").isNull()
).count()

print("Invalid reporting dates:", invalid_dates)

# ---------------------------------------
# Check 2: Negative attendance counts
# ---------------------------------------

invalid_attendances = df_silver.filter(
    F.col("number_of_attendances").isNull() |
    (F.col("number_of_attendances") < 0)
).count()

print("Invalid attendance counts:", invalid_attendances)

# ---------------------------------------
# Check 3: Triage total reconciliation
# ---------------------------------------

# Sum categories 1–5 for each facility and quarter
triage_totals = (
    df_silver
    .filter(F.col("triage_category").isin(
        "1", "2", "3", "4", "5"
    ))
    .groupBy("facility_code", "reporting_quarter")
    .agg(
        F.sum("number_of_attendances")
        .alias("calculated_total"),
        F.countDistinct("triage_category")
        .alias("category_count")
    )
)

# Published ALL totals
published_totals = (
    df_silver
    .filter(F.col("triage_category") == "ALL")
    .select(
        "facility_code",
        "reporting_quarter",
        F.col("number_of_attendances")
        .alias("published_total")
    )
)

# Compare both totals
reconciliation = (
    triage_totals
    .join(
        published_totals,
        ["facility_code", "reporting_quarter"],
        "full_outer"
    )
    .withColumn(
        "difference",
        F.col("calculated_total") -
        F.col("published_total")
    )
)

# Find discrepancies, missing totals, and incomplete categories
issues = reconciliation.filter(
    F.col("calculated_total").isNull() |
    F.col("published_total").isNull() |
    (F.col("category_count") != 5) |
    (F.col("difference") != 0)
)

print("Reconciliation issues:", issues.count())

display(issues.limit(20))


StatementMeta(, 4abdc89a-5642-4909-949b-16224ba212a9, 10, Finished, Available, Finished, False)

Invalid reporting dates: 0
Invalid attendance counts: 0
Reconciliation issues: 0


SynapseWidget(Synapse.DataFrame, 1947d70d-8ae8-47b5-8a69-95cd01b0a70d)

In [9]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_ed_quarterly")

StatementMeta(, 4abdc89a-5642-4909-949b-16224ba212a9, 11, Finished, Available, Finished, False)

In [10]:

from pyspark.sql import functions as F

# 1. Check reporting dates
invalid_dates = df_silver.filter(
    F.col("quarter_end_month").isNull()
).count()

# 2. Check attendance counts
invalid_attendances = df_silver.filter(
    F.col("number_of_attendances").isNull() |
    (F.col("number_of_attendances") < 0)
).count()

# 3. Sum triage categories 1–5
triage_totals = (
    df_silver
    .filter(F.col("triage_category").isin(
        "1", "2", "3", "4", "5"
    ))
    .groupBy("facility_code", "reporting_quarter")
    .agg(
        F.sum("number_of_attendances")
        .alias("calculated_total"),
        F.countDistinct("triage_category")
        .alias("category_count")
    )
)

# 4. Get published ALL totals
published_totals = (
    df_silver
    .filter(F.col("triage_category") == "ALL")
    .select(
        "facility_code",
        "reporting_quarter",
        F.col("number_of_attendances")
        .alias("published_total")
    )
)

# 5. Compare totals
reconciliation = (
    triage_totals
    .join(
        published_totals,
        ["facility_code", "reporting_quarter"],
        "full_outer"
    )
    .withColumn(
        "difference",
        F.col("calculated_total") -
        F.col("published_total")
    )
)

issues = reconciliation.filter(
    F.col("calculated_total").isNull() |
    F.col("published_total").isNull() |
    (F.col("category_count") != 5) |
    (F.col("difference") != 0)
)

print("Invalid reporting dates:", invalid_dates)
print("Invalid attendance counts:", invalid_attendances)
print("Triage reconciliation issues:", issues.count())

display(issues.limit(20))


StatementMeta(, 4abdc89a-5642-4909-949b-16224ba212a9, 12, Finished, Available, Finished, False)

Invalid reporting dates: 0
Invalid attendance counts: 0
Triage reconciliation issues: 0


SynapseWidget(Synapse.DataFrame, eef28240-ac08-495c-8805-833c56228a78)

In [11]:

# Save the validated Silver DataFrame
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_ed_quarterly")

print("Silver Delta table saved successfully!")


StatementMeta(, 4abdc89a-5642-4909-949b-16224ba212a9, 13, Finished, Available, Finished, False)

Silver Delta table saved successfully!


In [12]:

# Read Silver table from Lakehouse
df_check = spark.table("silver_ed_quarterly")

print("Total rows:", df_check.count())
print("Total columns:", len(df_check.columns))

# Confirm date conversion
display(
    df_check.select(
        "reporting_quarter",
        "quarter_end_month"
    ).distinct().orderBy("quarter_end_month")
)


StatementMeta(, 4abdc89a-5642-4909-949b-16224ba212a9, 14, Finished, Available, Finished, False)

Total rows: 6930
Total columns: 18


SynapseWidget(Synapse.DataFrame, 6840a407-c23e-4027-b82c-4b14c16f70f1)